# 02 — Threshold Calibration and Out-of-Sample Backtest

End-to-end evaluation of the Multivariate Pair Trading Strategy (MPTS):

1. in-sample grid search of the z-score entry/exit thresholds;
2. rolling beta estimation for the beta-neutrality constraint;
3. daily out-of-sample backtest with optimal position sizing
   (Gurobi if licensed, SciPy SLSQP otherwise);
4. comparison against buy-and-hold benchmarks and trade-level statistics.

Run notebook 01 first (or `python scripts/download_data.py`) so that
`data/raw/prices.csv` exists.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

from src import backtest, data, hedging, metrics, signals, stat_tests

cfg = yaml.safe_load(open("../configs/default.yaml", encoding="utf-8"))
ANCHOR = cfg["universe"]["anchor"]
EQUITIES = cfg["universe"]["equities"]
WINDOW = cfg["signals"]["zscore_window"]

In [ ]:
prices = data.load_prices("../" + cfg["data"]["path"])
log_prices = data.to_log_prices(prices)
log_is, log_oos, split_date = data.train_test_split(
    log_prices, ANCHOR, cfg["split"]["ratio"]
)
RF = cfg["strategy"]["risk_free_rate"]
TC = cfg["strategy"]["transaction_cost"]

## 1. In-sample threshold calibration

Grid search over `open ∈ [1.0, 2.5)` and `close ∈ [0.5, 2.0)` (step 0.1,
`close < open` enforced), maximizing the average single-spread Sharpe across
the basket.

In [ ]:
grid_results, heatmap = signals.grid_search_thresholds(
    log_is, EQUITIES, ANCHOR, window=WINDOW, risk_free_rate=RF
)
open_grid = np.round(np.arange(1.0, 2.5, 0.1), 1)
close_grid = np.round(np.arange(0.5, 2.0, 0.1), 1)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(heatmap, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=close_grid, yticklabels=open_grid,
            linewidths=0.5, cbar_kws={"label": "avg Sharpe"}, ax=ax)
ax.set_xlabel("Closing threshold")
ax.set_ylabel("Opening threshold")
plt.tight_layout()
plt.show()

best_open = float(grid_results.iloc[0]["open"])
best_close = float(grid_results.iloc[0]["close"])
print(f"IS-optimal thresholds: open=±{best_open}, close=±{best_close}")

On the Bloomberg sample the optimum sits in a *stable region* of positive
Sharpe (open ±2.1 / close ±1.7) rather than an isolated spike — reassuring
against overfitting, but calibrated on a calmer 2016–2023 regime than the
explosive out-of-sample period. That regime mismatch is the story of this
project.

## 2. Hedging inputs: betas, covariances, mean-reversion speed

In [ ]:
FEATURES = [ANCHOR, cfg["universe"]["sector_proxy"]]

static_betas, residuals = hedging.estimate_static_betas(
    log_prices, EQUITIES, FEATURES, ANCHOR
)
display(static_betas.round(4))

anchor_returns = log_prices[ANCHOR].diff().dropna()
covariances = hedging.pair_covariances(
    anchor_returns, residuals.diff().dropna(), EQUITIES
)
rolling = hedging.rolling_betas(
    log_prices, EQUITIES, FEATURES, ANCHOR,
    window=cfg["strategy"]["rolling_beta_window"],
)

time_to_mean = signals.compute_time_to_mean(
    log_is, EQUITIES, ANCHOR, best_open, best_close, window=WINDOW
)
avg_ret_anchor = log_is[ANCHOR].diff().dropna().mean()
avg_ret_equity = {s: log_is[s].diff().dropna().mean() for s in EQUITIES}
print("Mean-reversion speed proxy (avg holding days, IS):")
for stock, days in time_to_mean.items():
    print(f"  {stock}/{ANCHOR}: {days:.1f}")

Controlling for the consumer-staples sector (`XLP`) isolates the residual
coffee sensitivity of each stock. Those betas are small — daily coffee moves
are secondary to sector dynamics for large caps — but they capture the
commodity-specific risk that the beta-neutrality constraint neutralizes.

## 3. Out-of-sample backtest — IS-optimized vs. manual thresholds

In [ ]:
configurations = {
    f"MPTS (IS-opt ±{best_open}/±{best_close})": (best_open, best_close),
    "MPTS (manual ±1.0/±0.6)": (1.0, 0.6),
}

results = {}
for label, (open_th, close_th) in configurations.items():
    ttm = signals.compute_time_to_mean(
        log_is, EQUITIES, ANCHOR, open_th, close_th, window=WINDOW
    )
    results[label] = backtest.run_strategy(
        log_prices, EQUITIES, ANCHOR, split_date,
        rolling, covariances, ttm, avg_ret_anchor, avg_ret_equity,
        open_threshold=open_th, close_threshold=close_th, window=WINDOW,
        risk_aversion=cfg["strategy"]["risk_aversion"],
        transaction_cost=TC, risk_free_rate=RF,
        solver=cfg["strategy"]["solver"],
    )
    print(f"{label}: Sharpe = {results[label].sharpe:.3f}, "
          f"{len(results[label].trades)} trades")

## 4. Benchmark comparison

In [ ]:
CAPITAL = cfg["backtest"]["initial_capital"]
kc1_value = backtest.buy_and_hold_benchmark(log_oos, [ANCHOR], CAPITAL, TC)
ew_value = backtest.buy_and_hold_benchmark(log_oos, EQUITIES, CAPITAL, TC)

fig, ax = plt.subplots(figsize=(12, 6))
for label, result in results.items():
    ax.plot(result.portfolio_value(CAPITAL), lw=1.3, label=label)
ax.plot(kc1_value, ls=":", color="saddlebrown", lw=1.3, label=f"B&H {ANCHOR}")
ax.plot(ew_value, ls=":", color="grey", lw=1.3, label="B&H EW basket")
ax.set_ylabel("Portfolio value ($)")
ax.legend()
ax.grid(alpha=0.4)
plt.tight_layout()
plt.show()

report = pd.DataFrame(
    {label: metrics.portfolio_metrics(result.portfolio_value(CAPITAL), RF)
     for label, result in results.items()}
    | {f"B&H {ANCHOR}": metrics.portfolio_metrics(kc1_value, RF),
       "B&H EW basket": metrics.portfolio_metrics(ew_value, RF)}
)
report.round(4)

## 5. Trade-level statistics

In [ ]:
pd.DataFrame(
    {label: metrics.trade_metrics(result.trades)
     for label, result in results.items()}
).round(4)

## Takeaways (Bloomberg sample, from the report)

* The IS-optimized thresholds (±2.1/±1.7), calibrated on a moderate-volatility
  regime, rarely triggered during the explosive 2023–2026 OOS window:
  Sharpe **−0.165**, only ~116 trades in three years.
* The framework itself worked as designed: annualized volatility compressed to
  **11.2%** vs **35.7%** for the KC1 buy-and-hold, and the rolling beta hedge
  neutralized the commodity exposure daily.
* Tighter manual thresholds (±1.0/±0.6, exploratory only — they embed
  look-ahead) lift the Sharpe to **0.349**, showing the failure mode is
  *threshold regime mismatch*, not the optimization machinery.
* Root cause: 3 of the 4 pairs are not cointegrated at 5%. Mean-reversion
  cannot be manufactured by an optimizer when the long-run equilibrium is
  statistically absent.